In [5]:
import pandas as pd
import numpy as np

# Load clean dataset
df = pd.read_csv('../thesis_dataset/clean_dataset.csv')
print(f"Loaded: {df.shape}")

DATA_PATH    = '../thesis_dataset/'
appearances  = pd.read_csv(DATA_PATH + 'appearances.csv')
valuations   = pd.read_csv(DATA_PATH + 'player_valuations.csv')
games        = pd.read_csv(DATA_PATH + 'games.csv')

valuations['date']   = pd.to_datetime(valuations['date'])
valuations['season'] = valuations['date'].dt.year

# Performance Features
# Rate stats normalised per 90 minutes
df['goals_per90']               = (df['goals']   / df['minutes_played']) * 90
df['assists_per90']             = (df['assists']  / df['minutes_played']) * 90
df['goal_contributions']        = df['goals'] + df['assists']
df['goal_contributions_per90']  = df['goals_per90'] + df['assists_per90']

print("\n── Performance features ──")
print(df[['goals_per90', 'assists_per90', 
          'goal_contributions']].describe())

# Contextual Features
# Age squared captures non-linear peak-decline career curve
df['age_squared'] = df['age'] ** 2

# Log transform market value — raw values are right-skewed
df['log_market_value'] = np.log(df['market_value'])

# Position dummies
position_dummies = pd.get_dummies(df['position'], prefix='pos')
df = pd.concat([df, position_dummies], axis=1)

# Foot dummies
foot_dummies = pd.get_dummies(df['foot'], prefix='foot')
df = pd.concat([df, foot_dummies], axis=1)

print("\n── Position distribution ──")
print(df['position'].value_counts())

print("\n── Log market value ──")
print(df['log_market_value'].describe())

# Weighted Goal Contributions
# Goals and assists weighted by actual opponent strength
# per match, using opponent club total market value relative
# to the league seasonal average.
#
# Formula:
#   weighted_gc = Σ (goals × opponent_strength)
#               + Σ (assists × opponent_strength × 0.5)
#
# Where:
#   opponent_strength = opponent_club_value / league_avg_club_value
#
# Rationale: A goal against a high-value club reflects greater
# quality of opposition and should contribute more to a player's
# perceived value than a goal against a weaker side.

# Filter PL appearances
pl_appearances = appearances[
    appearances['competition_id'] == 'GB1'
].copy()
pl_appearances['season'] = pd.to_datetime(
    pl_appearances['date']
).dt.year

# Get PL games with opponent info
pl_games = games[games['competition_id'] == 'GB1'][[
    'game_id', 'home_club_id', 'away_club_id'
]].copy()

# Join appearances to games
pl_app_games = pl_appearances.merge(pl_games, on='game_id', how='inner')

# Determine opponent club per appearance
pl_app_games['opponent_id'] = pl_app_games.apply(
    lambda x: x['away_club_id'] 
    if x['player_club_id'] == x['home_club_id'] 
    else x['home_club_id'],
    axis=1
)

# Build opponent club value per season
opponent_value = valuations.groupby(
    ['current_club_id', 'season']
).agg(
    opponent_club_value = ('market_value_in_eur', 'sum')
).reset_index()
opponent_value.columns = ['opponent_id', 'season', 'opponent_club_value']

# League average club value per season
league_avg = valuations.groupby('season').agg(
    league_avg_club_value = ('market_value_in_eur', 'mean')
).reset_index()

# Join opponent value and league average
pl_app_games = pl_app_games.merge(
    opponent_value, on=['opponent_id', 'season'], how='left'
)
pl_app_games = pl_app_games.merge(league_avg, on='season', how='left')

# Fill missing opponent values with league average (neutral weight)
pl_app_games['opponent_club_value'] = pl_app_games[
    'opponent_club_value'
].fillna(pl_app_games['league_avg_club_value'])

# Calculate opponent strength and weighted gc per appearance
pl_app_games['opponent_strength'] = (
    pl_app_games['opponent_club_value'] / 
    pl_app_games['league_avg_club_value']
)

pl_app_games['weighted_gc'] = (
    pl_app_games['goals']   * pl_app_games['opponent_strength'] +
    pl_app_games['assists'] * pl_app_games['opponent_strength'] * 0.5
)

# Aggregate per player per season
weighted_stats = pl_app_games.groupby(['player_id', 'season']).agg(
    weighted_gc           = ('weighted_gc', 'sum'),
    avg_opponent_strength = ('opponent_strength', 'mean')
).reset_index()

# Join back to main dataframe
df = df.merge(weighted_stats, on=['player_id', 'season'], how='left')

print("\n── Weighted GC ──")
print(df[['goals', 'weighted_gc', 
          'avg_opponent_strength']].describe())

# Final dataset summary
print(f"\nFinal dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nNull counts:\n{df.isnull().sum()}")

# Save engineered dataset
df.to_csv('../thesis_dataset/engineered_dataset.csv', index=False)
print("\nSaved: ../thesis_dataset/engineered_dataset.csv")

Loaded: (3819, 14)

── Performance features ──
       goals_per90  assists_per90  goal_contributions
count  3819.000000    3819.000000         3819.000000
mean      0.134896       0.106564            5.080126
std       0.169233       0.112166            5.724912
min       0.000000       0.000000            0.000000
25%       0.000000       0.000000            1.000000
50%       0.075125       0.076078            3.000000
75%       0.190510       0.164974            7.000000
max       1.547912       0.761905           46.000000

── Position distribution ──
position
Defender    1572
Midfield    1246
Attack      1001
Name: count, dtype: int64

── Log market value ──
count    3819.000000
mean       16.203789
std         1.093773
min        12.429216
25%        15.424948
50%        16.300417
75%        17.034386
max        19.008467
Name: log_market_value, dtype: float64

── Weighted GC ──
             goals   weighted_gc  avg_opponent_strength
count  3819.000000   3819.000000            38